# Stage 2 — Data Preprocessing

This notebook turns the raw LinkedIn Job Postings 2023-2024 tables into one clean, salary-tiered table, `data/processed/cleaned_jobs.csv`, with these columns:

`job_id, job_title, company_name, location, experience_level, industry, skills_list, normalized_salary, salary_tier`

It also:
- cleans the 1.3M LinkedIn Jobs & Skills skill strings into per-job lists (`data/processed/linkedin_jobs_skills.csv`) for association mining
- compares the salary tiers against the Data Science Salaries dataset
- writes a summary to `outputs/02_summary.txt`

The logic lives in `src/preprocessing.py`. This notebook calls it one step at a time so each intermediate result can be inspected. To run everything without the notebook: `python -m src.preprocessing`.

## Step 1 — Setup

Find the project root, add it to `sys.path` so `src` can be imported, and send the module's log messages to the notebook.

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import preprocessing as pp

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stats = {}

## Step 2 — Load postings and join the related tables

`postings.csv` is the main table, with one row per `job_id`. It gets three left joins:
- **`jobs/salaries.csv`**: fills any salary field that postings leaves empty. In the current data both files hold identical salary values, so this fills 0 values. It's kept as a consistency check.
- **`jobs/job_skills.csv` + `mappings/skills.csv`**: skill codes (e.g. `IT`) become names, collected into a list per job.
- **`jobs/job_industries.csv` + `mappings/industries.csv`**: industry names per job, joined with `"; "` (a job can have up to 3).

In [ ]:
df = pp.load_postings(RAW_DIR)
stats["postings_rows"] = len(df)
before = len(df)
df = df.drop_duplicates("job_id")
stats["dropped_duplicate_job_id"] = before - len(df)

df, stats["salary_values_from_salaries_csv"] = pp.attach_salaries(df, RAW_DIR)
df = pp.attach_skills(df, RAW_DIR)
df = pp.attach_industries(df, RAW_DIR)
df[["job_id", "title", "skills_list", "industry", "normalized_salary", "pay_period"]].head()

## Step 3 — Normalize titles and experience level

Titles are lowercased and whitespace is collapsed. A missing `formatted_experience_level` (about 24% of postings) becomes `"Unknown"` so it can still be grouped and one-hot encoded later.

In [ ]:
df = pp.normalize_titles(df)
df["experience_level"] = df["formatted_experience_level"].fillna("Unknown")
df["experience_level"].value_counts()

## Step 4 — Annualize salaries

`normalized_salary` is the main salary field. Where it's missing but a raw salary exists, it's computed as the midpoint of `min_salary`/`max_salary` (or `med_salary`) times the `pay_period` multiplier:

| pay_period | multiplier |
|---|---|
| HOURLY | 2080 |
| WEEKLY | 52 |
| BIWEEKLY | 26 |
| MONTHLY | 12 |
| YEARLY | 1 |

This matches how the dataset computes `normalized_salary` itself: on every row that has both, the formula reproduces it exactly. The dataset already fills `normalized_salary` for every posting with a salary, so this step currently fills 0 values. It's there in case a future data refresh leaves gaps.

In [ ]:
df, stats["salary_values_annualized"] = pp.annualize_salary(df)
df["pay_period"].value_counts(dropna=False)

## Step 5 — Filter salary rows

Rows are removed in this order:
1. **No salary.** Both `normalized_salary` and `min_salary` are null. This is about 71% of postings, because most LinkedIn postings don't list pay.
2. **Non-USD.** `normalized_salary` isn't converted between currencies, so mixing currencies would distort the tiers.
3. **Implausible values.** Annual salaries outside `[$10k, $1M]` are data-entry errors: $0, hourly rates tagged as yearly, and one $535M posting. Change the bounds with `pp.MIN_PLAUSIBLE_SALARY` / `pp.MAX_PLAUSIBLE_SALARY`.

In [ ]:
df = pp.filter_salary_rows(df, stats)
{k: stats[k] for k in ["dropped_no_salary", "dropped_non_usd", "dropped_implausible_salary"]}

## Step 6 — Impute remaining missing salaries

Any salary still missing is filled with the median for its `experience_level`. A group with no known salaries falls back to the overall median. After Step 5 no salaries are missing, so this fills 0 values today.

In [ ]:
df, stats["salary_values_imputed"] = pp.impute_salary_by_experience(df)
df["normalized_salary"].describe()

## Step 7 — Discretize salary into tiers

The 33rd and 66th percentiles of `normalized_salary` split the jobs into three roughly equal groups: **Low** (bottom third), **Mid**, and **High** (top third). The cross-tab by experience level is a quick sanity check: Directors and Executives should be mostly High, and Interns mostly Low.

In [ ]:
df["salary_tier"], stats["tier_thresholds"] = pp.assign_salary_tier(df["normalized_salary"])
df["normalized_salary"] = df["normalized_salary"].round(2)
stats["final_rows"] = len(df)
cleaned = df[pp.OUTPUT_COLUMNS].reset_index(drop=True)

p33, p66 = stats["tier_thresholds"]
print(f"Low <= ${p33:,.0f} < Mid <= ${p66:,.0f} < High")
display(cleaned["salary_tier"].value_counts())
(pd.crosstab(cleaned["experience_level"], cleaned["salary_tier"], normalize="index")
   .reindex(columns=pp.TIER_LABELS) * 100).round(1)

## Step 8 — Skill lists

Each job's `skills_list` is a Python list of lowercased, de-duplicated skill names.

⚠️ **Limitation:** `mappings/skills.csv` has only **35 job-function categories** (e.g. *information technology*, *sales*, *engineering*), not technical skills like *python* or *sql*. Jobs average under 2 of them. That's enough as a classification feature, but too coarse for meaningful association rules or skill-based clustering.

In [ ]:
skill_counts = cleaned["skills_list"].explode().value_counts()
print(f"Distinct skills: {len(skill_counts)} | mean per job: {cleaned['skills_list'].str.len().mean():.2f}")
skill_counts.head(15)

## Step 9 — Clean 1.3M LinkedIn Jobs skill lists

`linkedin_jobs/job_skills.csv` stores each job's skills as one comma-separated string, keyed by `job_link`. It can't be joined to the postings table because the two datasets share no key. The skills are fine-grained (e.g. *python*, *patient care*, *data analysis*), so they're cleaned into lists and saved separately as a transaction source for Stage 4 association mining.

In [ ]:
jobs_skills = pp.load_linkedin_jobs_skills(RAW_DIR)
print(f"{len(jobs_skills):,} jobs, mean {jobs_skills['skills_list'].str.len().mean():.1f} skills per job")
jobs_skills["skills_list"].explode().value_counts().head(20)

## Step 10 — Salary benchmark against Data Science Salaries

The LinkedIn tier thresholds are compared with US full-time rows from the Data Science Salaries dataset. Data science pay is expected to sit mostly in the LinkedIn High tier, since the LinkedIn data covers all occupations.

In [ ]:
benchmark = pp.benchmark_ds_salaries(RAW_DIR, stats["tier_thresholds"], cleaned)
ds33, ds66 = benchmark["ds_thresholds"]
print(f"ds_salaries 33rd / 66th percentile: ${ds33:,.0f} / ${ds66:,.0f}")
print("Share of ds_salaries rows per LinkedIn tier:", benchmark["ds_share_in_linkedin_tiers"])
benchmark["median_by_experience"]

## Step 11 — Save outputs

- `data/processed/cleaned_jobs.csv`: the main cleaned table. `skills_list` is saved as a list literal. Read it back with `pp.load_cleaned_jobs(path)`, which parses it into Python lists.
- `data/processed/linkedin_jobs_skills.csv`: `job_link, skills_list` for about 1.3M jobs.
- `outputs/02_summary.txt`: the preprocessing summary.

In [ ]:
cleaned_path = PROCESSED_DIR / "cleaned_jobs.csv"
cleaned.to_csv(cleaned_path, index=False)
jobs_skills.to_csv(PROCESSED_DIR / "linkedin_jobs_skills.csv", index=False)

summary = pp.build_summary(cleaned, jobs_skills, stats, benchmark)
(OUTPUT_DIR / "02_summary.txt").write_text(summary, encoding="utf-8")

reloaded = pp.load_cleaned_jobs(cleaned_path)
assert list(reloaded.columns) == pp.OUTPUT_COLUMNS
assert isinstance(reloaded.loc[0, "skills_list"], list)
print(f"Saved {len(cleaned):,} rows to {cleaned_path}")

## Next steps

- Stage 3 loads `cleaned_jobs.csv` into the SQLite star schema.
- Stage 4 should mine rules from `linkedin_jobs_skills.csv`, which has fine-grained skills. The 35 categories in `cleaned_jobs.csv` are too coarse.
- For Stage 5, consider extracting technical skills from the postings `description` text, using the most frequent skills in `linkedin_jobs_skills.csv` as the keyword list. That would give TF-IDF richer features than the 35 categories.